In [1]:
import json
import os
import pandas as pd

def load_json_data(file_path):
    """Reads and returns data from a JSON file."""
    if not os.path.exists(file_path):
        print(f"Error: File '{file_path}' not found in the current directory.")
        return None

    with open(file_path, 'r') as file:
        return json.load(file)

def inspect_csv_data(file_path):
    df = pd.read_csv(file_path)
    num_columns = df.shape[1]
    return num_columns

def compute_num_params_nn(all_layers, layer_in, layer_out):
    total_params = 0
    total_layers = [layer_in] + all_layers + [layer_out]
    for i in range(len(total_layers) - 1):
        n_in = total_layers[i]
        n_out = total_layers[i+1]

        # Calculation formula: (Inputs * Outputs) + Outputs
        weights = n_in * n_out
        biases = n_out
        subtotal = weights + biases
        total_params += subtotal
    return total_params

In [2]:
"""Extracts and prints key metrics from the specified JSON files."""
# Define file names
name_list = [
        'AgNP',
        'CO2RRLCA',
        'CO2RRNPV',
        'CO2RRMSP',
        'CO2HEx10',
        'CO2HPx10',
]

data_performance = {'name': [], 'model': [], 'R2': [], 'num_params': []}
for name in name_list:
    dir_project = r"D:\pykan\github\workflows\Hyein"
    mlp_file = os.path.join("material_nn_models", name, f"{name}_metrics.json")
    kan_file = os.path.join("material_kan_models", name, f"{name}_kan_metrics.json")
    data_path = os.path.join(dir_project, 'data', f"{name}.csv")

    # Load data
    mlp_data = load_json_data(mlp_file)
    kan_data = load_json_data(kan_file)

    n_cols = inspect_csv_data(data_path)
    n_in = n_cols - 1
    n_out = 1

    # Process Standard Neural Network Data
    if mlp_data:
        nn_layers = mlp_data.get('best_params').get("hidden_layer_sizes")
        print(nn_layers)
        nn_params = compute_num_params_nn(nn_layers, n_in, n_out)

        data_performance['name'].append(name)
        data_performance['model'].append('NN')
        data_performance['R2'].append(mlp_data.get('test_r2', 0))
        data_performance['num_params'].append(nn_params)

    # Process KAN / Symbolic Regression Data
    if kan_data:
        kan_params = kan_data.get('best_params')
        kan_params = n_in * n_out * (kan_params['grid'] + kan_params['k'] + 1)

        data_performance['name'].append(name)
        data_performance['model'].append('KAN')
        data_performance['R2'].append(kan_data.get('test_r2', 0))
        data_performance['num_params'].append(kan_params)
df_performance = pd.DataFrame(data_performance)
print(df_performance)

[64, 32, 16]
[128]
[64, 64]
[128, 64]
[128]
[64, 64]
        name model        R2  num_params
0       AgNP    NN  0.941127        3009
1       AgNP   KAN  0.812329          70
2   CO2RRLCA    NN  0.982547        1281
3   CO2RRLCA   KAN  0.965041          72
4   CO2RRNPV    NN  0.994874        4801
5   CO2RRNPV   KAN  0.971202         112
6   CO2RRMSP    NN  0.989171        9473
7   CO2RRMSP   KAN  0.915158         112
8   CO2HEx10    NN  0.974814        2305
9   CO2HEx10   KAN  0.883803         144
10  CO2HPx10    NN  0.828834        5313
11  CO2HPx10   KAN  0.858984         144


In [14]:
name = 'CO2HPx10'
seed_list = [i for i in range(10)]

data_performance = {'name': [], 'seed': [], 'model': [], 'R2': [], 'num_params': []}
for seed in seed_list:
    dir_project = r"D:\pykan\github\workflows\Hyein"
    mlp_file = os.path.join("material_nn_models", name+f"_seed_{seed}", f"{name}_metrics.json")
    kan_file = os.path.join("material_kan_models", name+f"_seed_{seed}", f"{name}_kan_metrics.json")
    data_path = os.path.join(dir_project, 'data', f"{name}.csv")

    # Load data
    mlp_data = load_json_data(mlp_file)
    kan_data = load_json_data(kan_file)

    n_cols = inspect_csv_data(data_path)
    n_in = n_cols - 1
    n_out = 1

    # Process Standard Neural Network Data
    if mlp_data:
        nn_layers = mlp_data.get('best_params').get("hidden_layer_sizes")
        print(nn_layers)
        nn_params = compute_num_params_nn(nn_layers, n_in, n_out)

        data_performance['name'].append(name)
        data_performance['seed'].append(seed)
        data_performance['model'].append('NN')
        data_performance['R2'].append(mlp_data.get('test_r2', 0))
        data_performance['num_params'].append(nn_params)

    # Process KAN / Symbolic Regression Data
    if kan_data:
        kan_params = kan_data.get('best_params')
        kan_params = n_in * n_out * (kan_params['grid'] + kan_params['k'] + 1)

        data_performance['name'].append(name)
        data_performance['seed'].append(seed)
        data_performance['model'].append('KAN')
        data_performance['R2'].append(kan_data.get('test_r2', 0))
        data_performance['num_params'].append(kan_params)
df_performance = pd.DataFrame(data_performance)
print(df_performance)

       name  seed model        R2  num_params
0  CO2HPx10     0    NN  0.834723        3713
1  CO2HPx10     1    NN  0.835082        3713
2  CO2HPx10     2    NN  0.844408        3985
3  CO2HPx10     3    NN  0.840174        3985
4  CO2HPx10     4    NN  0.839968        3985
5  CO2HPx10     5    NN  0.840036        3985
6  CO2HPx10     6    NN  0.827737       10497
7  CO2HPx10     7    NN  0.840024        3985
8  CO2HPx10     8    NN  0.828249       10497
9  CO2HPx10     9    NN  0.840271        3985
